In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from PIL import Image


In [8]:
# Resolve paths from the project config so the dataset location is defined in one place.
working_dir = Path.cwd().resolve()
PROJECT_ROOT = working_dir if (working_dir / 'config' / 'config.yaml').is_file() else working_dir.parent
with (PROJECT_ROOT / 'config' / 'config.yaml').open(encoding='utf-8') as config_file:
    config = yaml.safe_load(config_file)

DATA_DIR = Path(config['data_ingestion']['source_dir']).expanduser()
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Dataset directory not found: {DATA_DIR}')

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg'}
class_dirs = [path for path in sorted(DATA_DIR.iterdir()) if path.is_dir()]
if len(class_dirs) < 2:
    raise ValueError(f'Expected at least two class folders in {DATA_DIR}')

image_table = pd.DataFrame(
    (
        {'image_path': image_path, 'class_name': class_dir.name, 'label': label}
        for label, class_dir in enumerate(class_dirs)
        for image_path in sorted(class_dir.iterdir())
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS
    )
)
if image_table.empty:
    raise ValueError(f'No supported images found in {DATA_DIR}')

class_names = tuple(class_dir.name for class_dir in class_dirs)
image_table.groupby('class_name', as_index=False).size().rename(columns={'size': 'image_count'})


,class_name,image_count
0,adenocarcinoma,195
1,normal,148


In [9]:
# Load every CT image as RGB, resize it for a CNN, and keep labels aligned with images.
IMAGE_SIZE = (224, 224)

images = np.stack([
    np.asarray(Image.open(image_path).convert('RGB').resize(IMAGE_SIZE), dtype=np.uint8)
    for image_path in image_table['image_path']
])
labels = image_table['label'].to_numpy(dtype=np.int64)

print(f'Classes (label order): {dict(enumerate(class_names))}')
print(f'Loaded images shape: {images.shape}; labels shape: {labels.shape}')
print(f'Pixel range: {images.min()} to {images.max()}')

Classes (label order): {0: 'adenocarcinoma', 1: 'normal'}
Loaded images shape: (343, 224, 224, 3); labels shape: (343,)
Pixel range: 0 to 255


In [11]:
import tensorflow as tf 

2026-08-14 18:24:52.545994: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [12]:
from pathlib import Path 
from dataclasses import dataclass 
@dataclass(frozen= True )

class PrepareBaseModelConfig : 
    root_dir : Path 
    base_model_path : Path 
    updated_base_model_path : Path 
    params_image_size : list 
    params_learning_rate: float 
    params_include_top: bool 
    params_weights: str 
    params_classes: int 
    

In [14]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories 

In [17]:
class ConfigurationManager:
    def __init__(self, config_filepath=None, params_filepath=None):
        working_dir = Path.cwd().resolve()
        project_root = working_dir if (working_dir / 'config' / 'config.yaml').is_file() else working_dir.parent
        self.project_root = project_root
        config_filepath = Path(config_filepath or project_root / 'config' / 'config.yaml')
        params_filepath = Path(params_filepath or project_root / 'params.yaml')

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        artifacts_root = Path(self.config.artifacts_root).expanduser()
        self.artifacts_root = (artifacts_root if artifacts_root.is_absolute() else project_root / artifacts_root).resolve()
        create_directories([self.artifacts_root])

    def _resolve_path(self, path_value):
        path = Path(path_value).expanduser()
        return (path if path.is_absolute() else self.project_root / path).resolve()
        
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig : 
        config= self.config.prepare_base_model
        root_dir = self._resolve_path(config.root_dir)
        create_directories([root_dir])
        prepare_base_model_config= PrepareBaseModelConfig(
            root_dir=root_dir,
            base_model_path=self._resolve_path(config.base_model_path),
            updated_base_model_path=self._resolve_path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES 
        
        )
        return prepare_base_model_config

In [25]:
import os 
import tensorflow as tf 

In [27]:
class PrepareBaseModel:
    def __init__(self,config: PrepareBaseModelConfig):
        self.config=config 
    
    def get_base_model(self):
        self.model=tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size, 
            weights=self.config.params_weights,
            include_top= self.config.params_include_top
        )
        self.save_model(path=self.config.base_model_path,model=self.model)
        
        
        
    @staticmethod 
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [28]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)


    
    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
    

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)
    


    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)




In [30]:
try:
    config=ConfigurationManager()
    prepare_base_model_config=config.get_prepare_base_model_config()
    prepare_base_model=PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e :
    raise e 

[2026-08-14 20:22:41]: INFO: yaml file: /Users/Shared/ThoraxGuard/config/config.yaml loaded successfully
[2026-08-14 20:22:41]: INFO: yaml file: /Users/Shared/ThoraxGuard/params.yaml loaded successfully
[2026-08-14 20:22:41]: INFO: created directory at: /Users/Shared/ThoraxGuard/artifacts
[2026-08-14 20:22:41]: INFO: created directory at: /Users/Shared/ThoraxGuard/artifacts/prepare_base_model
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [43]:
from dataclasses import dataclass 
from pathlib import Path 
@dataclass(frozen=True)
class TrainingConfig:
    root_dir :Path 
    trained_model_path:Path 
    updated_base_model_path:Path 
    training_data:Path 
    params_epochs:int 
    params_batch_size:int 
    params_is_augmentation:bool 
    params_image_size:list 
    

In [45]:
from cnnClassifier.constants import * 
from cnnClassifier.utils.common import read_yaml,create_directories 
import tensorflow as tf 

In [46]:
class ConfigurationManager:
    def __init__(self, config_filepath=None, params_filepath=None):
        working_dir = Path.cwd().resolve()
        project_root = working_dir if (working_dir / 'config' / 'config.yaml').is_file() else working_dir.parent
        self.project_root = project_root
        config_filepath = Path(config_filepath or project_root / 'config' / 'config.yaml')
        params_filepath = Path(params_filepath or project_root / 'params.yaml')

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        artifacts_root = Path(self.config.artifacts_root).expanduser()
        self.artifacts_root = (artifacts_root if artifacts_root.is_absolute() else project_root / artifacts_root).resolve()
        create_directories([self.artifacts_root])

    def _resolve_path(self, path_value):
        path = Path(path_value).expanduser()
        return (path if path.is_absolute() else self.project_root / path).resolve()
        
    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig : 
        config= self.config.prepare_base_model
        root_dir = self._resolve_path(config.root_dir)
        create_directories([root_dir])
        prepare_base_model_config= PrepareBaseModelConfig(
            root_dir=root_dir,
            base_model_path=self._resolve_path(config.base_model_path),
            updated_base_model_path=self._resolve_path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES 
        
        )
        return prepare_base_model_config

    def get_training_config(self) -> TrainingConfig:
        config = self.config.training
        root_dir = self._resolve_path(config.root_dir)
        create_directories([root_dir])
        training_config = TrainingConfig(
            root_dir=root_dir,
            trained_model_path=self._resolve_path(config.trained_model_path),
            updated_base_model_path=self._resolve_path(config.updated_base_model_path),
            training_data=self._resolve_path(config.training_data),
            params_epochs=self.params.EPOCHS,
            params_batch_size=self.params.BATCH_SIZE,
            params_is_augmentation=self.params.AUGMENTATION,
            params_image_size=self.params.IMAGE_SIZE
        )
        return training_config

In [47]:
import os 
import urllib.request as request 
import time 

In [48]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )



In [50]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-08-15 10:55:09]: INFO: yaml file: /Users/Shared/ThoraxGuard/config/config.yaml loaded successfully
[2026-08-15 10:55:09]: INFO: yaml file: /Users/Shared/ThoraxGuard/params.yaml loaded successfully
[2026-08-15 10:55:09]: INFO: created directory at: /Users/Shared/ThoraxGuard/artifacts
[2026-08-15 10:55:09]: INFO: created directory at: /Users/Shared/ThoraxGuard/artifacts/training
Found 68 images belonging to 2 classes.
Found 275 images belonging to 2 classes.
Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.5174 - loss: 15.8470 - val_accuracy: 1.0000 - val_loss: 0.0122
Epoch 2/10
 1/17 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.8750 - loss: 0.3877

/Users/Shared/ThoraxGuard/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 321ms/step - accuracy: 0.8750 - loss: 0.3877 - val_accuracy: 0.7188 - val_loss: 0.5715
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.5135 - loss: 14.6908 - val_accuracy: 0.8594 - val_loss: 2.0868
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 325ms/step - accuracy: 0.5625 - loss: 5.1875 - val_accuracy: 0.3906 - val_loss: 17.9068
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - accuracy: 0.6100 - loss: 10.9586 - val_accuracy: 0.6406 - val_loss: 2.4321
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 346ms/step - accuracy: 0.8125 - loss: 1.0753 - val_accuracy: 1.0000 - val_loss: 3.5382e-04
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - accuracy: 0.6718 - loss: 6.4852 - val_accuracy: 0.9844 - val_loss: 0.1100
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 344ms/step - accuracy: 0.9375 - loss: 1.1041 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 9/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - accuracy: 0.7375 - loss: 5.1714 - val_accuracy: 0.8281 - val_l

In [ ]:
os.environ["ML_FLOW_TRACKING_URI"]="https://dagshub.com/shovo896/ThoraxGuard.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="shovo896"
os.environ["MLFLOW_TRACKIMG_PASSWORD"]=""

In [52]:
! pip install dagshub

  Obtaining dependency information for dagshub from https://files.pythonhosted.org/packages/f8/a0/4d63eeef1873d18063fddfabcf92d8a7e6cda7cdd3695eb6060ba413c76b/dagshub-0.7.1-py3-none-any.whl.metadata
  Obtaining dependency information for appdirs>=1.4.4 from https://files.pythonhosted.org/packages/3b/00/2344469e2084fb287c2e0b57b72910309874c3245463acd6cf5e3db69324/appdirs-1.4.4-py2.py3-none-any.whl.metadata
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Obtaining dependency information for GitPython>=3.1.29 from https://files.pythonhosted.org/packages/ef/ed/ae57eb7d344f43f87b74b3a281ead6ec7d6394eef72a7b1dcb28dd089550/gitpython-3.1.59-py3-none-any.whl.metadata
  Obtaining dependency information for dacite~=1.6.0 from https://files.pythonhosted.org/packages/06/9d/11a073172d889e9e0d0ad270a1b468876c82d759af7864a8095dfc73f46d/dacite-1.6.0-py3-none-any.whl.metadata
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Obtaining dependency information for tenaci

In [55]:
!pip install mlflow

  Obtaining dependency information for mlflow from https://files.pythonhosted.org/packages/5a/ff/24e530a328b3b2c9d004e94cdd59ea5ec9c052c4935799f675d7bb1f1d2c/mlflow-3.15.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 468.6 kB/s eta 0:00:00a 0:00:01
  Obtaining dependency information for mlflow-skinny==3.15.1 from https://files.pythonhosted.org/packages/f3/9e/4e8138583321f6dfbaa35360f46056b4ecc06d505d130c03a5d81aec5620/mlflow_skinny-3.15.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
  Obtaining dependency information for mlflow-tracing==3.15.1 from https://files.pythonhosted.org/packages/f1/c7/13db54bd1b6525a0f09d880260d9dcf8072ae89155dae7a74c9e75a1ba03/mlflow_tracing-3.15.1-py3-none-any.whl.metadata
  Obtaining dependency information for aiohttp<4,>=3.7.0 from https://files.pythonhosted.org/packages/88/11/e7a70a209eb9a067c0d3212b518a0134e3484f5178c7533878b6b514d469/aiohttp-3.14.3-cp312-

In [54]:
import dagshub
dagshub.init(repo_owner='shovo896', repo_name='ThoraxGuard', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

[2026-08-15 15:20:14]: INFO: HTTP Request: GET https://dagshub.com/api/v1/repos/shovo896/ThoraxGuard "HTTP/1.1 200 OK"


Initialized MLflow to track repo "shovo896/ThoraxGuard"

[2026-08-15 15:20:14]: INFO: Initialized MLflow to track repo "shovo896/ThoraxGuard"


Repository shovo896/ThoraxGuard initialized!

[2026-08-15 15:20:14]: INFO: Repository shovo896/ThoraxGuard initialized!


ModuleNotFoundError: No module named 'mlflow'